In [1]:
# ============================================================
# WeReview - Hybrid Model
# Step 1: Environment and path verification
# ============================================================

import os
import numpy as np
import pandas as pd
import torch

print("=" * 60)
print("WEREVIEW HYBRID MODEL - SETUP")
print("=" * 60)

# Project paths
PROJECT_ROOT = ".."

BEHAVIORAL_DATA = "../datasets/public_reviews_dataset_cleaned.csv"
DISTILBERT_MODEL = "../models/distilbert_fake_review"

print("\nChecking required files...")

print("Behavioral dataset exists:",
      os.path.exists(BEHAVIORAL_DATA))

print("DistilBERT model folder exists:",
      os.path.exists(DISTILBERT_MODEL))

print("\nPyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\nSetup check complete.")

WEREVIEW HYBRID MODEL - SETUP

Checking required files...
Behavioral dataset exists: True
DistilBERT model folder exists: True

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 2050

Setup check complete.


In [2]:
# ============================================================
# Step 2: Inspect the saved DistilBERT model
# ============================================================

import os

print("=" * 60)
print("SAVED DISTILBERT MODEL FILES")
print("=" * 60)

for root, dirs, files in os.walk(DISTILBERT_MODEL):
    for file in files:
        path = os.path.join(root, file)
        size_mb = os.path.getsize(path) / (1024 ** 2)
        print(f"{file:<40} {size_mb:>8.2f} MB")

SAVED DISTILBERT MODEL FILES
config.json                                  0.00 MB
model.safetensors                          255.43 MB
optimizer.pt                               510.91 MB
rng_state.pth                                0.01 MB
scaler.pt                                    0.00 MB
scheduler.pt                                 0.00 MB
tokenizer.json                               0.68 MB
tokenizer_config.json                        0.00 MB
trainer_state.json                           0.03 MB
training_args.bin                            0.00 MB


In [5]:
# ============================================================
# Step 3: Load the original DistilBERT tokenizer + saved model
# ============================================================

from transformers import (
    AutoTokenizer,
    DistilBertForSequenceClassification
)
from safetensors.torch import load_file
import os
import torch

print("=" * 60)
print("LOADING SAVED DISTILBERT")
print("=" * 60)

# ------------------------------------------------------------
# 1. Load the original DistilBERT tokenizer
# ------------------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(
    "distilbert-base-uncased"
)

print("\nTokenizer loaded successfully!")


# ------------------------------------------------------------
# 2. Create the original DistilBERT architecture
# ------------------------------------------------------------

text_model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

print("DistilBERT architecture created!")


# ------------------------------------------------------------
# 3. Load YOUR trained model weights
# ------------------------------------------------------------

MODEL_WEIGHTS = os.path.join(
    DISTILBERT_MODEL,
    "model.safetensors"
)

print("\nLoading trained weights from:")
print(MODEL_WEIGHTS)

state_dict = load_file(MODEL_WEIGHTS)

text_model.load_state_dict(state_dict)

print("Trained weights loaded successfully!")


# ------------------------------------------------------------
# 4. Move model to GPU / CPU
# ------------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

text_model = text_model.to(device)

# Evaluation mode
text_model.eval()


# ------------------------------------------------------------
# 5. Display verification information
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("MODEL LOADED SUCCESSFULLY!")
print("=" * 60)

print("Model type:", text_model.__class__.__name__)
print("Number of labels:", text_model.config.num_labels)
print("Label mapping:", text_model.config.id2label)
print("Tokenizer:", tokenizer.name_or_path)
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("=" * 60)

LOADING SAVED DISTILBERT

Tokenizer loaded successfully!


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 376.37it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBERT architecture created!

Loading trained weights from:
../models/distilbert_fake_review\model.safetensors


FileNotFoundError: No such file or directory: ../models/distilbert_fake_review\model.safetensors

In [6]:
# ============================================================
# Step 3A: Find the saved DistilBERT files
# ============================================================

import os

print("=" * 60)
print("SEARCHING FOR SAVED DISTILBERT FILES")
print("=" * 60)

model_root = os.path.abspath("../models")

print("\nModels directory:")
print(model_root)

print("\nContents:\n")

for root, dirs, files in os.walk(model_root):
    level = root.replace(model_root, "").count(os.sep)
    indent = "    " * level
    print(f"{indent}{os.path.basename(root)}/")

    for file in files:
        print(f"{indent}    {file}")

print("\n" + "=" * 60)

SEARCHING FOR SAVED DISTILBERT FILES

Models directory:
c:\Users\USER\OneDrive\Desktop\WeReview_DistilBERT\models

Contents:

models/
    distilbert_fake_review/
        checkpoint-4041/
        checkpoint-8082/
            config.json
            model.safetensors
            optimizer.pt
            rng_state.pth
            scaler.pt
            scheduler.pt
            tokenizer.json
            tokenizer_config.json
            trainer_state.json
            training_args.bin
    distilbert_fake_review_final/
        config.json
        model.safetensors
        tokenizer.json
        tokenizer_config.json
        training_args.bin
    distilbert_lievgarcia_fake_review/
        checkpoint-2100/
        checkpoint-4200/
            config.json
            model.safetensors
            optimizer.pt
            rng_state.pth
            scaler.pt
            scheduler.pt
            trainer_state.json
            training_args.bin
    lievgarcia_smoke_test/
        checkpoint-2/
    

In [7]:
# ============================================================
# Step 3: Load the trained DistilBERT model
# ============================================================

import os
import torch
from transformers import AutoTokenizer, DistilBertForSequenceClassification
from safetensors.torch import load_file

print("=" * 60)
print("LOADING TRAINED DISTILBERT")
print("=" * 60)

# ------------------------------------------------------------
# 1. Correct model folder
# ------------------------------------------------------------

DISTILBERT_MODEL = "../models/distilbert_fake_review_final"

MODEL_WEIGHTS = os.path.join(
    DISTILBERT_MODEL,
    "model.safetensors"
)

print("\nModel folder:")
print(os.path.abspath(DISTILBERT_MODEL))

print("\nWeights file:")
print(os.path.abspath(MODEL_WEIGHTS))

print("\nWeights file exists:",
      os.path.exists(MODEL_WEIGHTS))


# ------------------------------------------------------------
# 2. Load the original tokenizer
# ------------------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(
    "distilbert-base-uncased"
)

print("\nTokenizer loaded successfully!")


# ------------------------------------------------------------
# 3. Create DistilBERT architecture
# ------------------------------------------------------------

text_model = DistilBertForSequenceClassification(
    DistilBertForSequenceClassification.config_class.from_pretrained(
        "distilbert-base-uncased",
        num_labels=2
    )
)

print("DistilBERT architecture created!")


# ------------------------------------------------------------
# 4. Load trained weights from model.safetensors
# ------------------------------------------------------------

print("\nLoading trained weights...")

state_dict = load_file(
    MODEL_WEIGHTS,
    device="cpu"
)

text_model.load_state_dict(
    state_dict,
    strict=True
)

print("Trained weights loaded successfully!")


# ------------------------------------------------------------
# 5. Move model to GPU / CPU
# ------------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

text_model = text_model.to(device)

text_model.eval()


# ------------------------------------------------------------
# 6. Final verification
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("DISTILBERT LOADED SUCCESSFULLY!")
print("=" * 60)

print("Model type:",
      text_model.__class__.__name__)

print("Number of labels:",
      text_model.config.num_labels)

print("Device:",
      device)

if torch.cuda.is_available():
    print("GPU:",
          torch.cuda.get_device_name(0))

print("=" * 60)

LOADING TRAINED DISTILBERT

Model folder:
c:\Users\USER\OneDrive\Desktop\WeReview_DistilBERT\models\distilbert_fake_review_final

Weights file:
c:\Users\USER\OneDrive\Desktop\WeReview_DistilBERT\models\distilbert_fake_review_final\model.safetensors

Weights file exists: True

Tokenizer loaded successfully!
DistilBERT architecture created!

Loading trained weights...
Trained weights loaded successfully!

DISTILBERT LOADED SUCCESSFULLY!
Model type: DistilBertForSequenceClassification
Number of labels: 2
Device: cuda
GPU: NVIDIA GeForce RTX 2050


In [8]:
# ============================================================
# Step 4: Load behavioral dataset
# ============================================================

BEHAVIORAL_DATA = "../datasets/public_reviews_dataset_cleaned.csv"

behavior_df = pd.read_csv(BEHAVIORAL_DATA)

print("=" * 60)
print("BEHAVIORAL DATASET LOADED")
print("=" * 60)

print("\nDataset shape:", behavior_df.shape)

print("\nColumns:")
print(behavior_df.columns.tolist())

print("\nFirst 3 rows:")
display(behavior_df.head(3))

print("\nDataset loaded successfully!")

C:\Users\USER\AppData\Local\Temp\ipykernel_9116\1042292973.py:7: DtypeWarning: Columns (16: fake_review_campaign_start_date) have mixed types. Specify dtype option on import or set low_memory=False.
  behavior_df = pd.read_csv(BEHAVIORAL_DATA)


BEHAVIORAL DATASET LOADED

Dataset shape: (381734, 23)

Columns:
['asin', 'review_id', 'reviewer_id', 'review_title', 'review_text', 'review_rating', 'review_date', 'product_title', 'product_url', 'number_of_helpful', 'number_of_photos', 'photo_thumbnail_urls', 'photo_fullsize_urls', 'asin_url', 'review_url', 'reviewer_url', 'fake_review_campaign_start_date', 'fake_review_product', 'reviewer_classified_fake', 'reviewer_classified_honest', 'reviewer_labeled_fake', 'reviewer_labeled_honest', 'review_is_removed_by_amazon']

First 3 rows:


,asin,review_id,reviewer_id,review_title,review_text,review_rating,review_date,product_title,product_url,number_of_helpful,...,asin_url,review_url,reviewer_url,fake_review_campaign_start_date,fake_review_product,reviewer_classified_fake,reviewer_classified_honest,reviewer_labeled_fake,reviewer_labeled_honest,review_is_removed_by_amazon
0,B003MROHZW,R2RDK8SL1QXN8I,AHAHNBDF4GJ26DAL3VS7MGJDNCKA,Dog muzzle,Serves it purpose.,5,2016-01-22,Downtown Pet Supply Quick Fit Dog Muzzle with ...,https://www.amazon.com/Small-Muzzle-Downtown-P...,0.0,...,https://www.amazon.com/dp/B003MROHZW,https://www.amazon.com/gp/customer-reviews/R2R...,https://www.amazon.com/gp/profile/amzn1.accoun...,2020-02-08,True,True,False,0.0,0.0,0.0
1,B008DZ6JPO,R1CSQAGJT0JXO7,AEZ2DKBYDRPE5O3ENPHV3XRJBMGQ,Awesome product!!,I use this ALL the time!! Looks really nice ...,5,2017-09-25,Winsome 92436 Luggage Rack with Shelf,https://www.amazon.com/Winsome-92436-Luggage-R...,0.0,...,https://www.amazon.com/dp/B008DZ6JPO,https://www.amazon.com/gp/customer-reviews/R1C...,https://www.amazon.com/gp/profile/amzn1.accoun...,NaN,True,True,False,0.0,0.0,0.0
2,B008DZ6JPO,R1XD30XZ3BXO7F,AF52D4YPSCPGLWNYVXZDGG525R2A,Good quality. Color is nice just like the pict...,Well made product. I purchased that refurbishe...,5,2018-11-14,Winsome 92436 Luggage Rack with Shelf,https://www.amazon.com/Winsome-92436-Luggage-R...,0.0,...,https://www.amazon.com/dp/B008DZ6JPO,https://www.amazon.com/gp/customer-reviews/R1X...,https://www.amazon.com/gp/profile/amzn1.accoun...,NaN,True,True,False,0.0,0.0,0.0



Dataset loaded successfully!


In [9]:
# ============================================================
# Step 5: Recreate the behavioral train/validation/test split
# ============================================================

from sklearn.model_selection import train_test_split
import numpy as np

print("=" * 60)
print("CREATING BEHAVIORAL DATA SPLITS")
print("=" * 60)

# Keep only reviews with an explicit reviewer-level label
behavior_df = behavior_df[
    (behavior_df["reviewer_labeled_fake"] == 1) |
    (behavior_df["reviewer_labeled_honest"] == 1)
].copy()

# Create target
# 1 = fake
# 0 = genuine
behavior_df["target"] = np.where(
    behavior_df["reviewer_labeled_fake"] == 1,
    1,
    0
)

print("\nBehavioral dataset shape:", behavior_df.shape)

print("\nTarget distribution:")
print(behavior_df["target"].value_counts())

# ------------------------------------------------------------
# Same 70 / 15 / 15 stratified split used for XGBoost
# ------------------------------------------------------------

train_df, temp_df = train_test_split(
    behavior_df,
    test_size=0.30,
    stratify=behavior_df["target"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["target"],
    random_state=42
)

print("\n" + "=" * 60)
print("SPLIT SIZES")
print("=" * 60)

print("Training:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

print("\nTest target distribution:")
print(test_df["target"].value_counts())

print("\nTest fake percentage:",
      round(test_df["target"].mean() * 100, 2), "%")

print("\nBehavioral split recreated successfully!")

CREATING BEHAVIORAL DATA SPLITS

Behavioral dataset shape: (32940, 24)

Target distribution:
target
1    22138
0    10802
Name: count, dtype: int64

SPLIT SIZES
Training: (23058, 24)
Validation: (4941, 24)
Test: (4941, 24)

Test target distribution:
target
1    3320
0    1621
Name: count, dtype: int64

Test fake percentage: 67.19 %

Behavioral split recreated successfully!


In [10]:
# ============================================================
# Step 6: Prepare behavioral test reviews for DistilBERT
# ============================================================

print("=" * 60)
print("PREPARING TEST REVIEWS FOR DISTILBERT")
print("=" * 60)

# Copy the behavioral test set
hybrid_test = test_df.copy()

# Make sure review text is a string
hybrid_test["review_text"] = (
    hybrid_test["review_text"]
    .fillna("")
    .astype(str)
)

# Reset index so predictions align perfectly
hybrid_test = hybrid_test.reset_index(drop=True)

print("\nNumber of test reviews:", len(hybrid_test))

print("Missing review texts:",
      hybrid_test["review_text"].eq("").sum())

print("\nFirst 3 review texts:\n")

for i in range(min(3, len(hybrid_test))):
    print(f"Review {i + 1}:")
    print(hybrid_test.loc[i, "review_text"][:300])
    print("-" * 60)

print("\nText preparation complete!")

PREPARING TEST REVIEWS FOR DISTILBERT

Number of test reviews: 4941
Missing review texts: 0

First 3 review texts:

Review 1:
finally got around to trying this scale. the app on my phone exceeded my expectations. it is so user friendly and gives so much info. I highly recommend this! the scale is very accurate also, I tested it with weights!
------------------------------------------------------------
Review 2:
  Fell apart. I really liked it, but the little screws kept coming out. It was no big deal the first time because I just screwed it back in and made sure to make it as tight as possible. After the third time, I couldn't use this anymore. Having my watch fall off while I'm doing things is annoying an
------------------------------------------------------------
Review 3:
  these did not work, kept sliding out, painful to put in
------------------------------------------------------------

Text preparation complete!


In [11]:
# ============================================================
# Step 7: Generate DistilBERT predictions
# ============================================================

from torch.utils.data import DataLoader, TensorDataset
import torch
import numpy as np

print("=" * 60)
print("GENERATING DISTILBERT PREDICTIONS")
print("=" * 60)

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

BATCH_SIZE = 16
MAX_LENGTH = 256

texts = hybrid_test["review_text"].tolist()

print("\nNumber of reviews:", len(texts))
print("Batch size:", BATCH_SIZE)
print("Max token length:", MAX_LENGTH)
print("Device:", device)

# ------------------------------------------------------------
# Tokenize all reviews
# ------------------------------------------------------------

print("\nTokenizing reviews...")

encodings = tokenizer(
    texts,
    truncation=True,
    padding=True,
    max_length=MAX_LENGTH,
    return_tensors="pt"
)

print("Tokenization complete!")

print("Input IDs shape:",
      encodings["input_ids"].shape)

# ------------------------------------------------------------
# Move through GPU in batches
# ------------------------------------------------------------

input_ids = encodings["input_ids"]
attention_mask = encodings["attention_mask"]

all_probabilities = []
all_predictions = []

print("\nRunning DistilBERT inference...")

with torch.no_grad():

    for start in range(0, len(texts), BATCH_SIZE):

        end = min(
            start + BATCH_SIZE,
            len(texts)
        )

        batch_input_ids = input_ids[start:end].to(device)
        batch_attention_mask = attention_mask[start:end].to(device)

        outputs = text_model(
            input_ids=batch_input_ids,
            attention_mask=batch_attention_mask
        )

        probabilities = torch.softmax(
            outputs.logits,
            dim=1
        )

        predictions = torch.argmax(
            probabilities,
            dim=1
        )

        all_probabilities.extend(
            probabilities[:, 1].cpu().numpy()
        )

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        if end % 500 < BATCH_SIZE or end == len(texts):
            print(
                f"Processed {end}/{len(texts)} reviews"
            )

# ------------------------------------------------------------
# Store predictions
# ------------------------------------------------------------

hybrid_test["distilbert_fake_probability"] = np.array(
    all_probabilities
)

hybrid_test["distilbert_prediction"] = np.array(
    all_predictions
)

print("\n" + "=" * 60)
print("DISTILBERT INFERENCE COMPLETE")
print("=" * 60)

print("\nPrediction count:",
      len(hybrid_test["distilbert_prediction"]))

print(
    "Fake probability range:",
    round(hybrid_test["distilbert_fake_probability"].min(), 4),
    "to",
    round(hybrid_test["distilbert_fake_probability"].max(), 4)
)

print("\nFirst 10 predictions:")

display(
    hybrid_test[
        [
            "target",
            "distilbert_fake_probability",
            "distilbert_prediction"
        ]
    ].head(10)
)

print("\nDistilBERT predictions generated successfully!")

GENERATING DISTILBERT PREDICTIONS

Number of reviews: 4941
Batch size: 16
Max token length: 256
Device: cuda

Tokenizing reviews...
Tokenization complete!
Input IDs shape: torch.Size([4941, 256])

Running DistilBERT inference...
Processed 512/4941 reviews
Processed 1008/4941 reviews
Processed 1504/4941 reviews
Processed 2000/4941 reviews
Processed 2512/4941 reviews
Processed 3008/4941 reviews
Processed 3504/4941 reviews
Processed 4000/4941 reviews
Processed 4512/4941 reviews
Processed 4941/4941 reviews

DISTILBERT INFERENCE COMPLETE

Prediction count: 4941
Fake probability range: 0.0 to 1.0

First 10 predictions:


,target,distilbert_fake_probability,distilbert_prediction
0,1,0.000023,0
1,0,0.000010,0
2,0,0.000014,0
3,0,0.000290,0
4,1,0.000294,0
5,1,0.000009,0
6,1,0.000012,0
7,1,0.000010,0
8,0,0.000020,0
9,0,0.000010,0



DistilBERT predictions generated successfully!


In [12]:
# ============================================================
# Step 8: Evaluate DistilBERT on behavioral test set
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    confusion_matrix,
    classification_report
)

print("=" * 60)
print("DISTILBERT PERFORMANCE ON BEHAVIORAL TEST SET")
print("=" * 60)

y_true = hybrid_test["target"].values
y_pred = hybrid_test["distilbert_prediction"].values

accuracy = accuracy_score(y_true, y_pred)
balanced_acc = balanced_accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)

print("\nAccuracy:          {:.4f} ({:.2f}%)".format(
    accuracy, accuracy * 100
))

print("Balanced Accuracy: {:.4f} ({:.2f}%)".format(
    balanced_acc, balanced_acc * 100
))

print("Precision:         {:.4f} ({:.2f}%)".format(
    precision, precision * 100
))

print("Recall:            {:.4f} ({:.2f}%)".format(
    recall, recall * 100
))

print("F1 Score:          {:.4f} ({:.2f}%)".format(
    f1, f1 * 100
))

print("\n" + "=" * 60)
print("CONFUSION MATRIX")
print("=" * 60)

cm = confusion_matrix(y_true, y_pred)

print(cm)

print("\nRows = Actual")
print("Columns = Predicted")
print("\n              Predicted")
print("              Genuine   Fake")
print("Actual Genuine    {:5d}   {:5d}".format(cm[0, 0], cm[0, 1]))
print("Actual Fake       {:5d}   {:5d}".format(cm[1, 0], cm[1, 1]))

print("\n" + "=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)

print(
    classification_report(
        y_true,
        y_pred,
        target_names=["Genuine", "Fake"],
        digits=4,
        zero_division=0
    )
)

DISTILBERT PERFORMANCE ON BEHAVIORAL TEST SET

Accuracy:          0.3131 (31.31%)
Balanced Accuracy: 0.4601 (46.01%)
Precision:         0.3724 (37.24%)
Recall:            0.0325 (3.25%)
F1 Score:          0.0598 (5.98%)

CONFUSION MATRIX
[[1439  182]
 [3212  108]]

Rows = Actual
Columns = Predicted

              Predicted
              Genuine   Fake
Actual Genuine     1439     182
Actual Fake        3212     108

CLASSIFICATION REPORT
              precision    recall  f1-score   support

     Genuine     0.3094    0.8877    0.4589      1621
        Fake     0.3724    0.0325    0.0598      3320

    accuracy                         0.3131      4941
   macro avg     0.3409    0.4601    0.2593      4941
weighted avg     0.3517    0.3131    0.1907      4941



In [13]:
# ============================================================
# Step 9: Verify DistilBERT label mapping
# ============================================================

print("=" * 60)
print("VERIFYING DISTILBERT LABEL MAPPING")
print("=" * 60)

print("\nModel id2label:")
print(text_model.config.id2label)

print("\nModel label2id:")
print(text_model.config.label2id)

print("\nCurrent probability columns:")
print("Column 0 =", "probability of label 0")
print("Column 1 =", "probability of label 1")

print("\n" + "=" * 60)

VERIFYING DISTILBERT LABEL MAPPING

Model id2label:
{0: 'LABEL_0', 1: 'LABEL_1'}

Model label2id:
{'LABEL_0': 0, 'LABEL_1': 1}

Current probability columns:
Column 0 = probability of label 0
Column 1 = probability of label 1



In [14]:
# ============================================================
# Step 10: Verify original training label mapping
# ============================================================

print("=" * 60)
print("VERIFYING ORIGINAL TRAINING LABEL MAPPING")
print("=" * 60)

print("\nThe hybrid model needs to know exactly what:")
print("0 = ?")
print("1 = ?")

print("\nPlease check the label mapping used during training.")

print("\nExpected mapping from our previous setup was:")
print("OR -> 0")
print("CG -> 1")

print("\nWe will NOT assume which one means Fake until")
print("we verify it from the original training setup.")

print("=" * 60)

VERIFYING ORIGINAL TRAINING LABEL MAPPING

The hybrid model needs to know exactly what:
0 = ?
1 = ?

Please check the label mapping used during training.

Expected mapping from our previous setup was:
OR -> 0
CG -> 1

We will NOT assume which one means Fake until
we verify it from the original training setup.


In [15]:
# ============================================================
# Step 10A: Verify Salminen label meanings
# ============================================================

import pandas as pd

SALMINEN_TRAIN = "../datasets/salminen/train.csv"

salminen_train = pd.read_csv(SALMINEN_TRAIN)

print("=" * 60)
print("SALMINEN LABEL CHECK")
print("=" * 60)

print("\nColumns:")
print(salminen_train.columns.tolist())

print("\nLabel distribution:")
print(salminen_train["label"].value_counts())

print("\nUnique labels:")
print(salminen_train["label"].unique())

print("\nSample rows by label:")
display(
    salminen_train[
        ["label", "text_"]
    ].groupby("label")
    .head(3)
)

SALMINEN LABEL CHECK

Columns:
['category', 'rating', 'label', 'text_']

Label distribution:
label
OR    16171
CG    16157
Name: count, dtype: int64

Unique labels:
<ArrowStringArray>
['OR', 'CG']
Length: 2, dtype: str

Sample rows by label:


,label,text_
0,OR,I was puzzled to see why this only has four st...
1,CG,One little topple sideways and it broke the pi...
2,OR,"This is a very boring movie. But, the Canada s..."
3,CG,My dog loved this toy! She kept pulling on it...
4,OR,I bought mainly to pocket carry my Springfield...
6,CG,"Like make fire, this make it an easy to carry ..."


In [16]:
# ============================================================
# Step 10B: Find the exact label mapping used during training
# ============================================================

import json

NOTEBOOK_PATH = "../notebooks/01_distilbert_training.ipynb"

with open(NOTEBOOK_PATH, "r", encoding="utf-8") as f:
    notebook = json.load(f)

print("=" * 60)
print("SEARCHING ORIGINAL TRAINING NOTEBOOK")
print("=" * 60)

keywords = [
    "label2id",
    "id2label",
    "OR",
    "CG",
    "label_mapping",
    "label_map"
]

found = False

for i, cell in enumerate(notebook["cells"]):

    source = "".join(cell.get("source", []))

    if any(keyword in source for keyword in keywords):

        print("\n" + "-" * 60)
        print("CELL", i)
        print("-" * 60)
        print(source[:5000])

        found = True

if not found:
    print("\nNo matching label-mapping code found.")

SEARCHING ORIGINAL TRAINING NOTEBOOK

------------------------------------------------------------
CELL 3
------------------------------------------------------------
label_map = {
    "OR": 0,
    "CG": 1
}

train_df["label_id"] = train_df["label"].map(label_map)
val_df["label_id"] = val_df["label"].map(label_map)
test_df["label_id"] = test_df["label"].map(label_map)

print("Train labels:")
print(train_df["label_id"].value_counts())

print("\nValidation labels:")
print(val_df["label_id"].value_counts())

print("\nTest labels:")
print(test_df["label_id"].value_counts())

print("\nMissing mapped labels:")
print(
    train_df["label_id"].isna().sum(),
    val_df["label_id"].isna().sum(),
    test_df["label_id"].isna().sum()
)

------------------------------------------------------------
CELL 35
------------------------------------------------------------
from transformers import AutoTokenizer, AutoModelForSequenceClassification

saved_model_path = "../models/distilbert_fake_review_final"

In [17]:
# ============================================================
# Step 11: Check saved XGBoost behavioral model
# ============================================================

import os

print("=" * 60)
print("SEARCHING FOR SAVED XGBOOST MODEL")
print("=" * 60)

MODELS_DIR = "../models"

for root, dirs, files in os.walk(MODELS_DIR):
    for file in files:
        if file.endswith((".pkl", ".joblib", ".json")):
            print(os.path.join(root, file))

SEARCHING FOR SAVED XGBOOST MODEL
../models\distilbert_fake_review\checkpoint-8082\config.json
../models\distilbert_fake_review\checkpoint-8082\tokenizer.json
../models\distilbert_fake_review\checkpoint-8082\tokenizer_config.json
../models\distilbert_fake_review\checkpoint-8082\trainer_state.json
../models\distilbert_fake_review_final\config.json
../models\distilbert_fake_review_final\tokenizer.json
../models\distilbert_fake_review_final\tokenizer_config.json
../models\distilbert_lievgarcia_fake_review\checkpoint-4200\config.json
../models\distilbert_lievgarcia_fake_review\checkpoint-4200\trainer_state.json
../models\lievgarcia_smoke_test\checkpoint-2\config.json
../models\lievgarcia_smoke_test\checkpoint-2\trainer_state.json


In [18]:
# ============================================================
# Step 12: Recreate the behavioral XGBoost model
# ============================================================

from xgboost import XGBClassifier

BEHAVIORAL_FEATURES = [
    "rating",
    "is_1_star",
    "is_5_star",
    "helpful_votes",
    "has_helpful_votes",
    "photo_count",
    "has_photos",
    "review_text_length",
    "review_word_count",
    "review_title_length",
    "review_title_word_count",
    "review_year",
    "review_month",
    "review_day_of_week"
]

print("=" * 60)
print("BEHAVIORAL MODEL CONFIGURATION")
print("=" * 60)

print("\nNumber of features:", len(BEHAVIORAL_FEATURES))
print("\nFeatures:")
for i, feature in enumerate(BEHAVIORAL_FEATURES, 1):
    print(f"{i:2d}. {feature}")

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

print("\nXGBoost model created successfully.")
print(xgb_model)

BEHAVIORAL MODEL CONFIGURATION

Number of features: 14

Features:
 1. rating
 2. is_1_star
 3. is_5_star
 4. helpful_votes
 5. has_helpful_votes
 6. photo_count
 7. has_photos
 8. review_text_length
 9. review_word_count
10. review_title_length
11. review_title_word_count
12. review_year
13. review_month
14. review_day_of_week

XGBoost model created successfully.
XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=5, max_leaves=None,
              min_child_weight=5, missing=nan

In [19]:
# ============================================================
# Step 13: Create behavioral features for hybrid test set
# ============================================================

print("=" * 60)
print("CREATING BEHAVIORAL FEATURES")
print("=" * 60)

def create_behavioral_features(df):
    data = pd.DataFrame(index=df.index)

    data["rating"] = pd.to_numeric(
        df["review_rating"], errors="coerce"
    ).fillna(0)

    data["is_1_star"] = (
        data["rating"] == 1
    ).astype(int)

    data["is_5_star"] = (
        data["rating"] == 5
    ).astype(int)

    data["helpful_votes"] = pd.to_numeric(
        df["number_of_helpful"], errors="coerce"
    ).fillna(0)

    data["has_helpful_votes"] = (
        data["helpful_votes"] > 0
    ).astype(int)

    data["photo_count"] = pd.to_numeric(
        df["number_of_photos"], errors="coerce"
    ).fillna(0)

    data["has_photos"] = (
        data["photo_count"] > 0
    ).astype(int)

    review_text = (
        df["review_text"]
        .fillna("")
        .astype(str)
    )

    review_title = (
        df["review_title"]
        .fillna("")
        .astype(str)
    )

    data["review_text_length"] = (
        review_text.str.len()
    )

    data["review_word_count"] = (
        review_text.str.split().str.len()
    )

    data["review_title_length"] = (
        review_title.str.len()
    )

    data["review_title_word_count"] = (
        review_title.str.split().str.len()
    )

    review_date = pd.to_datetime(
        df["review_date"],
        errors="coerce"
    )

    data["review_year"] = (
        review_date.dt.year.fillna(0)
    )

    data["review_month"] = (
        review_date.dt.month.fillna(0)
    )

    data["review_day_of_week"] = (
        review_date.dt.dayofweek.fillna(0)
    )

    return data[BEHAVIORAL_FEATURES]


X_test_behavior = create_behavioral_features(
    hybrid_test
)

y_test_behavior = hybrid_test["target"].values

print("\nFeature matrix shape:", X_test_behavior.shape)
print("Expected shape: (4941, 14)")

print("\nMissing values:")
print(X_test_behavior.isna().sum().sum())

print("\nFirst 5 rows:")
print(X_test_behavior.head())

print("\nTarget distribution:")
print(pd.Series(y_test_behavior).value_counts().sort_index())

print("\nFeature preparation complete.")

CREATING BEHAVIORAL FEATURES

Feature matrix shape: (4941, 14)
Expected shape: (4941, 14)

Missing values:
0

First 5 rows:
   rating  is_1_star  is_5_star  helpful_votes  has_helpful_votes  \
0       5          0          1            0.0                  0   
1       1          1          0            0.0                  0   
2       1          1          0            1.0                  1   
3       5          0          1            0.0                  0   
4       5          0          1            0.0                  0   

   photo_count  has_photos  review_text_length  review_word_count  \
0          0.0           0                 217                 40   
1          0.0           0                 473                 91   
2          0.0           0                  57                 11   
3          0.0           0                  48                  9   
4          0.0           0                  52                 12   

   review_title_length  review_title_word_coun

In [20]:
# ============================================================
# Step 14: Train XGBoost on behavioral training split
# ============================================================

print("=" * 60)
print("TRAINING BEHAVIORAL XGBOOST MODEL")
print("=" * 60)

# Create training features from the original training split
X_train_behavior = create_behavioral_features(train_df)
y_train_behavior = train_df["target"].values

print("\nTraining feature shape:", X_train_behavior.shape)
print("Training labels:", len(y_train_behavior))

print("\nTraining label distribution:")
print(
    pd.Series(y_train_behavior)
    .value_counts()
    .sort_index()
)

# Train
xgb_model.fit(
    X_train_behavior,
    y_train_behavior
)

print("\n" + "=" * 60)
print("XGBOOST TRAINING COMPLETE")
print("=" * 60)

TRAINING BEHAVIORAL XGBOOST MODEL

Training feature shape: (23058, 14)
Training labels: 23058

Training label distribution:
0     7561
1    15497
Name: count, dtype: int64

XGBOOST TRAINING COMPLETE


In [21]:
# ============================================================
# Step 15: Evaluate recreated XGBoost on behavioral test set
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

print("=" * 60)
print("XGBOOST - INDEPENDENT TEST EVALUATION")
print("=" * 60)

# Predictions
xgb_test_prob = xgb_model.predict_proba(
    X_test_behavior
)[:, 1]

xgb_test_pred = (
    xgb_test_prob >= 0.50
).astype(int)

# Metrics
xgb_accuracy = accuracy_score(
    y_test_behavior,
    xgb_test_pred
)

xgb_balanced_accuracy = balanced_accuracy_score(
    y_test_behavior,
    xgb_test_pred
)

xgb_precision = precision_score(
    y_test_behavior,
    xgb_test_pred,
    zero_division=0
)

xgb_recall = recall_score(
    y_test_behavior,
    xgb_test_pred,
    zero_division=0
)

xgb_f1 = f1_score(
    y_test_behavior,
    xgb_test_pred,
    zero_division=0
)

print(f"\nAccuracy          : {xgb_accuracy:.4f} ({xgb_accuracy:.2%})")
print(f"Balanced Accuracy : {xgb_balanced_accuracy:.4f}")
print(f"Precision         : {xgb_precision:.4f}")
print(f"Recall            : {xgb_recall:.4f}")
print(f"F1 Score          : {xgb_f1:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(
    y_test_behavior,
    xgb_test_pred
))

print("\nClassification Report:")
print(
    classification_report(
        y_test_behavior,
        xgb_test_pred,
        target_names=[
            "GENUINE",
            "FAKE"
        ],
        digits=4
    )
)

XGBOOST - INDEPENDENT TEST EVALUATION

Accuracy          : 0.8075 (80.75%)
Balanced Accuracy : 0.7629
Precision         : 0.8328
Recall            : 0.8928
F1 Score          : 0.8618

Confusion Matrix:
[[1026  595]
 [ 356 2964]]

Classification Report:
              precision    recall  f1-score   support

     GENUINE     0.7424    0.6329    0.6833      1621
        FAKE     0.8328    0.8928    0.8618      3320

    accuracy                         0.8075      4941
   macro avg     0.7876    0.7629    0.7725      4941
weighted avg     0.8032    0.8075    0.8032      4941



In [22]:
# ============================================================
# STEP 16: Generate XGBoost validation probabilities
# ============================================================

print("=" * 60)
print("PREPARING XGBOOST VALIDATION SET")
print("=" * 60)

# Create validation features
X_val_behavior = create_behavioral_features(val_df)
y_val_behavior = val_df["target"].values

print("\nValidation feature shape:", X_val_behavior.shape)
print("Validation labels:", len(y_val_behavior))

# Generate probabilities — NOT final predictions yet
xgb_val_prob = xgb_model.predict_proba(
    X_val_behavior
)[:, 1]

print("\nProbability range:")
print("Minimum:", xgb_val_prob.min())
print("Maximum:", xgb_val_prob.max())

print("\nValidation preparation complete.")

PREPARING XGBOOST VALIDATION SET

Validation feature shape: (4941, 14)
Validation labels: 4941

Probability range:
Minimum: 0.006422252
Maximum: 0.9902666

Validation preparation complete.


In [23]:
# ============================================================
# STEP 17: Find optimal XGBoost threshold
# ============================================================

from sklearn.metrics import accuracy_score

threshold_results = []

for threshold in np.arange(0.20, 0.81, 0.01):

    val_pred = (
        xgb_val_prob >= threshold
    ).astype(int)

    accuracy = accuracy_score(
        y_val_behavior,
        val_pred
    )

    threshold_results.append({
        "threshold": round(float(threshold), 2),
        "accuracy": accuracy
    })

threshold_df = pd.DataFrame(threshold_results)

best_threshold_row = threshold_df.loc[
    threshold_df["accuracy"].idxmax()
]

print("=" * 60)
print("THRESHOLD OPTIMIZATION - VALIDATION SET")
print("=" * 60)

print(
    f"\nBest threshold : "
    f"{best_threshold_row['threshold']:.2f}"
)

print(
    f"Best validation accuracy : "
    f"{best_threshold_row['accuracy']:.4f} "
    f"({best_threshold_row['accuracy']:.2%})"
)

print("\nTop 10 thresholds:")

print(
    threshold_df
    .sort_values("accuracy", ascending=False)
    .head(10)
    .to_string(index=False)
)

THRESHOLD OPTIMIZATION - VALIDATION SET

Best threshold : 0.50
Best validation accuracy : 0.8096 (80.96%)

Top 10 thresholds:
 threshold  accuracy
      0.50  0.809553
      0.53  0.809148
      0.52  0.808136
      0.51  0.808136
      0.48  0.807731
      0.49  0.807731
      0.54  0.807731
      0.46  0.806719
      0.55  0.806315
      0.56  0.806315


In [24]:
# ============================================================
# STEP 18: Add review writing-style features
# ============================================================

import re

def add_writing_style_features(base_df):
    data = create_behavioral_features(base_df).copy()

    text = (
        base_df["review_text"]
        .fillna("")
        .astype(str)
    )

    title = (
        base_df["review_title"]
        .fillna("")
        .astype(str)
    )

    # Basic character statistics
    data["punctuation_count"] = text.str.count(
        r"[^\w\s]"
    )

    data["exclamation_count"] = text.str.count("!")

    data["question_count"] = text.str.count(r"\?")

    data["comma_count"] = text.str.count(",")

    data["digit_count"] = text.str.count(r"\d")

    # Uppercase characters
    data["uppercase_count"] = text.apply(
        lambda x: sum(c.isupper() for c in x)
    )

    # Alphabetic characters
    data["letter_count"] = text.apply(
        lambda x: sum(c.isalpha() for c in x)
    )

    # Uppercase ratio
    data["uppercase_ratio"] = (
        data["uppercase_count"] /
        data["letter_count"].replace(0, 1)
    )

    # Digit ratio
    data["digit_ratio"] = (
        data["digit_count"] /
        data["review_text_length"].replace(0, 1)
    )

    # Average word length
    data["avg_word_length"] = (
        data["review_text_length"] /
        data["review_word_count"].replace(0, 1)
    )

    # Unique-word ratio
    data["unique_word_ratio"] = text.apply(
        lambda x: (
            len(set(x.lower().split())) /
            max(len(x.split()), 1)
        )
    )

    # Title-to-review length ratio
    data["title_text_length_ratio"] = (
        data["review_title_length"] /
        data["review_text_length"].replace(0, 1)
    )

    return data


X_train_style = add_writing_style_features(train_df)
X_val_style = add_writing_style_features(val_df)

print("=" * 60)
print("WRITING-STYLE FEATURES")
print("=" * 60)

print("\nTraining shape:", X_train_style.shape)
print("Validation shape:", X_val_style.shape)

print("\nNew features:")

original_features = set(BEHAVIORAL_FEATURES)

new_features = [
    col for col in X_train_style.columns
    if col not in original_features
]

for i, feature in enumerate(new_features, 1):
    print(f"{i:2d}. {feature}")

print("\nMissing values:")
print("Train:", X_train_style.isna().sum().sum())
print("Validation:", X_val_style.isna().sum().sum())

WRITING-STYLE FEATURES

Training shape: (23058, 26)
Validation shape: (4941, 26)

New features:
 1. punctuation_count
 2. exclamation_count
 3. question_count
 4. comma_count
 5. digit_count
 6. uppercase_count
 7. letter_count
 8. uppercase_ratio
 9. digit_ratio
10. avg_word_length
11. unique_word_ratio
12. title_text_length_ratio

Missing values:
Train: 0
Validation: 0


In [25]:
# ============================================================
# STEP 19: Train XGBoost with writing-style features
# ============================================================

print("=" * 60)
print("TRAINING XGBOOST WITH WRITING-STYLE FEATURES")
print("=" * 60)

xgb_style_model = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_style_model.fit(
    X_train_style,
    y_train_behavior
)

print("\nTraining complete.")

# Validation probabilities
style_val_prob = xgb_style_model.predict_proba(
    X_val_style
)[:, 1]

# Use the already-established 0.50 threshold
style_val_pred = (
    style_val_prob >= 0.50
).astype(int)

style_val_accuracy = accuracy_score(
    y_val_behavior,
    style_val_pred
)

style_val_precision = precision_score(
    y_val_behavior,
    style_val_pred,
    zero_division=0
)

style_val_recall = recall_score(
    y_val_behavior,
    style_val_pred,
    zero_division=0
)

style_val_f1 = f1_score(
    y_val_behavior,
    style_val_pred,
    zero_division=0
)

print("\n" + "=" * 60)
print("VALIDATION RESULTS")
print("=" * 60)

print(
    f"\nAccuracy : {style_val_accuracy:.4f} "
    f"({style_val_accuracy:.2%})"
)

print(
    f"Precision: {style_val_precision:.4f} "
    f"({style_val_precision:.2%})"
)

print(
    f"Recall   : {style_val_recall:.4f} "
    f"({style_val_recall:.2%})"
)

print(
    f"F1 Score : {style_val_f1:.4f} "
    f"({style_val_f1:.2%})"
)

TRAINING XGBOOST WITH WRITING-STYLE FEATURES

Training complete.

VALIDATION RESULTS

Accuracy : 0.8128 (81.28%)
Precision: 0.8331 (83.31%)
Recall   : 0.9021 (90.21%)
F1 Score : 0.8663 (86.63%)


In [26]:
# ============================================================
# STEP 20: Analyze feature importance
# ============================================================

importance_df = pd.DataFrame({
    "feature": X_train_style.columns,
    "importance": xgb_style_model.feature_importances_
})

importance_df = (
    importance_df
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

print("=" * 60)
print("XGBOOST FEATURE IMPORTANCE")
print("=" * 60)

print(
    importance_df.to_string(index=False)
)

print("\n" + "=" * 60)
print("TOP 10 FEATURES")
print("=" * 60)

print(
    importance_df.head(10).to_string(index=False)
)

XGBOOST FEATURE IMPORTANCE
                feature  importance
              is_5_star    0.228040
           letter_count    0.114131
            review_year    0.109074
                 rating    0.101375
           review_month    0.074040
      review_word_count    0.061657
            photo_count    0.047911
             has_photos    0.039510
          helpful_votes    0.018379
      unique_word_ratio    0.015971
              is_1_star    0.014912
         question_count    0.014218
     review_text_length    0.013697
            digit_count    0.013664
            digit_ratio    0.013272
        uppercase_ratio    0.013196
        uppercase_count    0.012881
        avg_word_length    0.011638
      punctuation_count    0.011568
      exclamation_count    0.011427
            comma_count    0.010963
    review_title_length    0.010636
title_text_length_ratio    0.010350
review_title_word_count    0.010250
      has_helpful_votes    0.008744
     review_day_of_week    0.008499



In [27]:
# ============================================================
# STEP 21: Check sentiment-analysis availability
# ============================================================

print("=" * 60)
print("CHECKING SENTIMENT ANALYSIS SETUP")
print("=" * 60)

try:
    from textblob import TextBlob

    print("\nTextBlob imported successfully.")

    test_text = "This product is excellent and amazing."

    sentiment = TextBlob(test_text).sentiment

    print("\nTest sentence:", test_text)
    print("Polarity:", sentiment.polarity)
    print("Subjectivity:", sentiment.subjectivity)

    print("\nSentiment setup is ready.")

except ImportError:
    print("\nTextBlob is NOT installed.")
    print("We will install it in the next step.")

CHECKING SENTIMENT ANALYSIS SETUP

TextBlob is NOT installed.
We will install it in the next step.


In [28]:
# ============================================================
# STEP 23: Verify TextBlob
# ============================================================

from textblob import TextBlob

test_text = "This product is excellent and amazing."

sentiment = TextBlob(test_text).sentiment

print("=" * 60)
print("TEXTBLOB VERIFICATION")
print("=" * 60)

print("\nTest sentence:")
print(test_text)

print("\nPolarity:", sentiment.polarity)
print("Subjectivity:", sentiment.subjectivity)

print("\nTextBlob is ready.")

TEXTBLOB VERIFICATION

Test sentence:
This product is excellent and amazing.

Polarity: 0.8
Subjectivity: 0.95

TextBlob is ready.


In [29]:
# ============================================================
# STEP 24: Add sentiment features
# ============================================================

from textblob import TextBlob

def add_sentiment_features(base_df):
    data = add_writing_style_features(base_df).copy()

    text = (
        base_df["review_text"]
        .fillna("")
        .astype(str)
    )

    print("Calculating sentiment features...")

    sentiments = text.apply(
        lambda x: TextBlob(x).sentiment
    )

    data["sentiment_polarity"] = sentiments.apply(
        lambda x: x.polarity
    )

    data["sentiment_subjectivity"] = sentiments.apply(
        lambda x: x.subjectivity
    )

    return data


print("=" * 60)
print("ADDING SENTIMENT FEATURES")
print("=" * 60)

X_train_sentiment = add_sentiment_features(train_df)
X_val_sentiment = add_sentiment_features(val_df)

print("\n" + "=" * 60)
print("SENTIMENT FEATURES CREATED")
print("=" * 60)

print("\nTraining shape:", X_train_sentiment.shape)
print("Validation shape:", X_val_sentiment.shape)

print("\nNew features:")
print("sentiment_polarity")
print("sentiment_subjectivity")

print("\nMissing values:")
print("Train:", X_train_sentiment.isna().sum().sum())
print("Validation:", X_val_sentiment.isna().sum().sum())

print("\nSentiment statistics:")
print(
    X_train_sentiment[
        ["sentiment_polarity", "sentiment_subjectivity"]
    ].describe()
)

ADDING SENTIMENT FEATURES
Calculating sentiment features...
Calculating sentiment features...

SENTIMENT FEATURES CREATED

Training shape: (23058, 28)
Validation shape: (4941, 28)

New features:
sentiment_polarity
sentiment_subjectivity

Missing values:
Train: 0
Validation: 0

Sentiment statistics:
       sentiment_polarity  sentiment_subjectivity
count        23058.000000            23058.000000
mean             0.304211                0.570122
std              0.238402                0.186204
min             -1.000000                0.000000
25%              0.161693                0.481746
50%              0.293393                0.580556
75%              0.437500                0.678333
max              1.000000                1.000000


In [30]:
# ============================================================
# STEP 25: Evaluate sentiment-enhanced XGBoost
# ============================================================

print("=" * 60)
print("TRAINING XGBOOST WITH SENTIMENT FEATURES")
print("=" * 60)

xgb_sentiment_model = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_sentiment_model.fit(
    X_train_sentiment,
    y_train_behavior
)

print("\nTraining complete.")

# Validation prediction
sentiment_val_prob = xgb_sentiment_model.predict_proba(
    X_val_sentiment
)[:, 1]

sentiment_val_pred = (
    sentiment_val_prob >= 0.50
).astype(int)

# Metrics
sentiment_accuracy = accuracy_score(
    y_val_behavior,
    sentiment_val_pred
)

sentiment_precision = precision_score(
    y_val_behavior,
    sentiment_val_pred,
    zero_division=0
)

sentiment_recall = recall_score(
    y_val_behavior,
    sentiment_val_pred,
    zero_division=0
)

sentiment_f1 = f1_score(
    y_val_behavior,
    sentiment_val_pred,
    zero_division=0
)

print("\n" + "=" * 60)
print("VALIDATION RESULTS")
print("=" * 60)

print(
    f"\nAccuracy : {sentiment_accuracy:.4f} "
    f"({sentiment_accuracy:.2%})"
)

print(
    f"Precision: {sentiment_precision:.4f} "
    f"({sentiment_precision:.2%})"
)

print(
    f"Recall   : {sentiment_recall:.4f} "
    f"({sentiment_recall:.2%})"
)

print(
    f"F1 Score : {sentiment_f1:.4f} "
    f"({sentiment_f1:.2%})"
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_val_behavior,
        sentiment_val_pred
    )
)

TRAINING XGBOOST WITH SENTIMENT FEATURES

Training complete.

VALIDATION RESULTS

Accuracy : 0.8144 (81.44%)
Precision: 0.8311 (83.11%)
Recall   : 0.9085 (90.85%)
F1 Score : 0.8681 (86.81%)

Confusion Matrix:
[[1007  613]
 [ 304 3017]]


In [31]:
# ============================================================
# STEP 26: Rating-Sentiment Interaction Features
# ============================================================

def add_rating_sentiment_features(base_df):
    data = add_sentiment_features(base_df).copy()

    rating = data["rating"]
    polarity = data["sentiment_polarity"]

    # Normalize rating to approximately [-1, 1]
    rating_scaled = (rating - 3) / 2

    # Basic interaction
    data["rating_sentiment_interaction"] = (
        rating_scaled * polarity
    )

    # Difference between normalized rating and sentiment
    data["rating_sentiment_gap"] = (
        abs(rating_scaled - polarity)
    )

    # Strongly positive rating but negative text
    data["positive_rating_negative_sentiment"] = (
        ((rating >= 4) & (polarity < 0))
        .astype(int)
    )

    # Low rating but positive text
    data["negative_rating_positive_sentiment"] = (
        ((rating <= 2) & (polarity > 0))
        .astype(int)
    )

    # Strength of sentiment regardless of direction
    data["sentiment_strength"] = abs(polarity)

    return data


print("=" * 60)
print("ADDING RATING-SENTIMENT INTERACTION FEATURES")
print("=" * 60)

X_train_interaction = add_rating_sentiment_features(train_df)
X_val_interaction = add_rating_sentiment_features(val_df)

print("\nTraining shape:", X_train_interaction.shape)
print("Validation shape:", X_val_interaction.shape)

print("\nNew features:")

previous_features = set(X_train_sentiment.columns)

new_features = [
    col for col in X_train_interaction.columns
    if col not in previous_features
]

for i, feature in enumerate(new_features, 1):
    print(f"{i}. {feature}")

print("\nMissing values:")
print("Train:", X_train_interaction.isna().sum().sum())
print("Validation:", X_val_interaction.isna().sum().sum())

ADDING RATING-SENTIMENT INTERACTION FEATURES
Calculating sentiment features...
Calculating sentiment features...

Training shape: (23058, 33)
Validation shape: (4941, 33)

New features:
1. rating_sentiment_interaction
2. rating_sentiment_gap
3. positive_rating_negative_sentiment
4. negative_rating_positive_sentiment
5. sentiment_strength

Missing values:
Train: 0
Validation: 0


In [32]:
# ============================================================
# STEP 27: Evaluate rating-sentiment interaction features
# ============================================================

print("=" * 60)
print("TRAINING XGBOOST WITH INTERACTION FEATURES")
print("=" * 60)

xgb_interaction_model = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_interaction_model.fit(
    X_train_interaction,
    y_train_behavior
)

print("\nTraining complete.")

# Validation prediction
interaction_val_prob = xgb_interaction_model.predict_proba(
    X_val_interaction
)[:, 1]

interaction_val_pred = (
    interaction_val_prob >= 0.50
).astype(int)

# Metrics
interaction_accuracy = accuracy_score(
    y_val_behavior,
    interaction_val_pred
)

interaction_precision = precision_score(
    y_val_behavior,
    interaction_val_pred,
    zero_division=0
)

interaction_recall = recall_score(
    y_val_behavior,
    interaction_val_pred,
    zero_division=0
)

interaction_f1 = f1_score(
    y_val_behavior,
    interaction_val_pred,
    zero_division=0
)

print("\n" + "=" * 60)
print("VALIDATION RESULTS")
print("=" * 60)

print(
    f"\nAccuracy : {interaction_accuracy:.4f} "
    f"({interaction_accuracy:.2%})"
)

print(
    f"Precision: {interaction_precision:.4f} "
    f"({interaction_precision:.2%})"
)

print(
    f"Recall   : {interaction_recall:.4f} "
    f"({interaction_recall:.2%})"
)

print(
    f"F1 Score : {interaction_f1:.4f} "
    f"({interaction_f1:.2%})"
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_val_behavior,
        interaction_val_pred
    )
)

TRAINING XGBOOST WITH INTERACTION FEATURES

Training complete.

VALIDATION RESULTS

Accuracy : 0.8144 (81.44%)
Precision: 0.8326 (83.26%)
Recall   : 0.9061 (90.61%)
F1 Score : 0.8678 (86.78%)

Confusion Matrix:
[[1015  605]
 [ 312 3009]]


In [33]:
# ============================================================
# STEP 28: Controlled XGBoost Hyperparameter Tuning
# ============================================================

from itertools import product
from sklearn.metrics import accuracy_score

print("=" * 60)
print("CONTROLLED XGBOOST HYPERPARAMETER TUNING")
print("=" * 60)

param_grid = {
    "max_depth": [3, 5, 7],
    "min_child_weight": [1, 5, 10],
    "learning_rate": [0.03, 0.05, 0.10]
}

results = []

combinations = list(product(
    param_grid["max_depth"],
    param_grid["min_child_weight"],
    param_grid["learning_rate"]
))

print(f"\nTotal configurations: {len(combinations)}")
print("Features used: 28")
print("Training samples:", len(X_train_sentiment))
print("Validation samples:", len(X_val_sentiment))

for run_number, (depth, child_weight, lr) in enumerate(combinations, 1):

    print(
        f"\n[{run_number:02d}/{len(combinations)}] "
        f"depth={depth}, "
        f"min_child_weight={child_weight}, "
        f"learning_rate={lr}"
    )

    model = XGBClassifier(
        n_estimators=300,
        max_depth=depth,
        learning_rate=lr,
        min_child_weight=child_weight,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train_sentiment,
        y_train_behavior,
        verbose=False
    )

    val_prob = model.predict_proba(
        X_val_sentiment
    )[:, 1]

    val_pred = (
        val_prob >= 0.50
    ).astype(int)

    accuracy = accuracy_score(
        y_val_behavior,
        val_pred
    )

    results.append({
        "max_depth": depth,
        "min_child_weight": child_weight,
        "learning_rate": lr,
        "accuracy": accuracy
    })

    print(
        f"    Validation accuracy: "
        f"{accuracy:.4f} ({accuracy:.2%})"
    )

tuning_results = (
    pd.DataFrame(results)
    .sort_values("accuracy", ascending=False)
    .reset_index(drop=True)
)

print("\n" + "=" * 60)
print("TOP CONFIGURATIONS")
print("=" * 60)

print(
    tuning_results.head(10).to_string(index=False)
)

best_params = tuning_results.iloc[0].to_dict()

print("\n" + "=" * 60)
print("BEST CONFIGURATION")
print("=" * 60)

print("max_depth        :", int(best_params["max_depth"]))
print("min_child_weight :", int(best_params["min_child_weight"]))
print("learning_rate    :", best_params["learning_rate"])
print(
    "Validation accuracy:",
    f"{best_params['accuracy']:.2%}"
)

CONTROLLED XGBOOST HYPERPARAMETER TUNING

Total configurations: 27
Features used: 28
Training samples: 23058
Validation samples: 4941

[01/27] depth=3, min_child_weight=1, learning_rate=0.03
    Validation accuracy: 0.8114 (81.14%)

[02/27] depth=3, min_child_weight=1, learning_rate=0.05
    Validation accuracy: 0.8140 (81.40%)

[03/27] depth=3, min_child_weight=1, learning_rate=0.1
    Validation accuracy: 0.8152 (81.52%)

[04/27] depth=3, min_child_weight=5, learning_rate=0.03
    Validation accuracy: 0.8118 (81.18%)

[05/27] depth=3, min_child_weight=5, learning_rate=0.05
    Validation accuracy: 0.8124 (81.24%)

[06/27] depth=3, min_child_weight=5, learning_rate=0.1
    Validation accuracy: 0.8152 (81.52%)

[07/27] depth=3, min_child_weight=10, learning_rate=0.03
    Validation accuracy: 0.8116 (81.16%)

[08/27] depth=3, min_child_weight=10, learning_rate=0.05
    Validation accuracy: 0.8130 (81.30%)

[09/27] depth=3, min_child_weight=10, learning_rate=0.1
    Validation accuracy: 

In [34]:
# ============================================================
# STEP 29: Train best XGBoost configuration
# ============================================================

print("=" * 60)
print("TRAINING BEST BEHAVIORAL XGBOOST")
print("=" * 60)

best_xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=3,
    learning_rate=0.10,
    min_child_weight=10,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

best_xgb_model.fit(
    X_train_sentiment,
    y_train_behavior
)

print("\nBest XGBoost model trained successfully.")

# Generate validation probabilities
best_xgb_val_prob = best_xgb_model.predict_proba(
    X_val_sentiment
)[:, 1]

print("\nValidation probability range:")
print("Minimum:", best_xgb_val_prob.min())
print("Maximum:", best_xgb_val_prob.max())

print("\nReady for hybrid evaluation.")

TRAINING BEST BEHAVIORAL XGBOOST

Best XGBoost model trained successfully.

Validation probability range:
Minimum: 0.0025738238
Maximum: 0.9936039

Ready for hybrid evaluation.


In [35]:
# ============================================================
# STEP 30: DistilBERT probabilities for validation set
# ============================================================

print("=" * 60)
print("GENERATING DISTILBERT VALIDATION PROBABILITIES")
print("=" * 60)

hybrid_val = val_df.copy()

hybrid_val["review_text"] = (
    hybrid_val["review_text"]
    .fillna("")
    .astype(str)
)

hybrid_val = hybrid_val.reset_index(drop=True)

texts_val = hybrid_val["review_text"].tolist()

print("\nValidation reviews:", len(texts_val))

# Tokenize
val_encodings = tokenizer(
    texts_val,
    truncation=True,
    padding=True,
    max_length=256,
    return_tensors="pt"
)

print(
    "Tokenized shape:",
    val_encodings["input_ids"].shape
)

# Generate probabilities
distilbert_val_prob = []

BATCH_SIZE = 16

with torch.no_grad():

    for start in range(
        0,
        len(texts_val),
        BATCH_SIZE
    ):

        end = min(
            start + BATCH_SIZE,
            len(texts_val)
        )

        batch = {
            key: value[start:end].to(device)
            for key, value in val_encodings.items()
        }

        outputs = text_model(**batch)

        probabilities = torch.softmax(
            outputs.logits,
            dim=1
        )

        distilbert_val_prob.extend(
            probabilities[:, 1]
            .cpu()
            .numpy()
        )

        if (
            start == 0 or
            end == len(texts_val) or
            (start // BATCH_SIZE) % 50 == 0
        ):
            print(
                f"Processed {end}/{len(texts_val)}"
            )

distilbert_val_prob = np.array(
    distilbert_val_prob
)

print("\n" + "=" * 60)
print("DISTILBERT VALIDATION PREDICTIONS READY")
print("=" * 60)

print(
    "Probability range:",
    distilbert_val_prob.min(),
    "to",
    distilbert_val_prob.max()
)

print(
    "Number of probabilities:",
    len(distilbert_val_prob)
)

print("\nExpected:", len(val_df))

GENERATING DISTILBERT VALIDATION PROBABILITIES

Validation reviews: 4941
Tokenized shape: torch.Size([4941, 256])
Processed 16/4941
Processed 816/4941
Processed 1616/4941
Processed 2416/4941
Processed 3216/4941
Processed 4016/4941
Processed 4816/4941
Processed 4941/4941

DISTILBERT VALIDATION PREDICTIONS READY
Probability range: 8.665925e-06 to 0.9999757
Number of probabilities: 4941

Expected: 4941


In [36]:
# ============================================================
# STEP 31: Hybrid Probability Blending
# ============================================================

print("=" * 60)
print("HYBRID MODEL - PROBABILITY BLENDING")
print("=" * 60)

y_val = y_val_behavior

hybrid_results = []

# XGBoost weight goes from 0.0 to 1.0
for xgb_weight in np.arange(0.0, 1.01, 0.05):

    distilbert_weight = 1.0 - xgb_weight

    # Combined fake probability
    hybrid_prob = (
        xgb_weight * best_xgb_val_prob
        +
        distilbert_weight * distilbert_val_prob
    )

    # Standard threshold
    hybrid_pred = (
        hybrid_prob >= 0.50
    ).astype(int)

    accuracy = accuracy_score(
        y_val,
        hybrid_pred
    )

    precision = precision_score(
        y_val,
        hybrid_pred,
        zero_division=0
    )

    recall = recall_score(
        y_val,
        hybrid_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_val,
        hybrid_pred,
        zero_division=0
    )

    hybrid_results.append({
        "xgb_weight": round(float(xgb_weight), 2),
        "distilbert_weight": round(
            float(distilbert_weight), 2
        ),
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

hybrid_results_df = (
    pd.DataFrame(hybrid_results)
    .sort_values("accuracy", ascending=False)
    .reset_index(drop=True)
)

print("\n" + "=" * 60)
print("HYBRID RESULTS - SORTED BY ACCURACY")
print("=" * 60)

print(
    hybrid_results_df.to_string(index=False)
)

best_hybrid = hybrid_results_df.iloc[0]

print("\n" + "=" * 60)
print("BEST HYBRID CONFIGURATION")
print("=" * 60)

print(
    f"XGBoost weight     : "
    f"{best_hybrid['xgb_weight']:.2f}"
)

print(
    f"DistilBERT weight  : "
    f"{best_hybrid['distilbert_weight']:.2f}"
)

print(
    f"Validation accuracy: "
    f"{best_hybrid['accuracy']:.2%}"
)

print(
    f"Precision          : "
    f"{best_hybrid['precision']:.2%}"
)

print(
    f"Recall             : "
    f"{best_hybrid['recall']:.2%}"
)

print(
    f"F1                 : "
    f"{best_hybrid['f1']:.2%}"
)

HYBRID MODEL - PROBABILITY BLENDING

HYBRID RESULTS - SORTED BY ACCURACY
 xgb_weight  distilbert_weight  accuracy  precision   recall       f1
       1.00               0.00  0.815422   0.831727 0.909365 0.868815
       0.95               0.05  0.815220   0.836877 0.900632 0.867585
       0.90               0.10  0.812791   0.842481 0.887383 0.864350
       0.85               0.15  0.808338   0.847890 0.871123 0.859349
       0.80               0.20  0.803481   0.855414 0.851551 0.853478
       0.75               0.25  0.795790   0.863751 0.826558 0.844745
       0.70               0.30  0.784254   0.875208 0.791930 0.831489
       0.65               0.35  0.755110   0.883400 0.732310 0.800790
       0.60               0.40  0.702287   0.891286 0.634447 0.741249
       0.55               0.45  0.569318   0.900604 0.403794 0.557588
       0.50               0.50  0.335762   0.602094 0.034628 0.065490
       0.45               0.55  0.332524   0.557214 0.033725 0.063600
       0.40      

In [37]:
# STEP 1: Reviewer behavioral features

def add_reviewer_features(train_df, val_df, test_df):
    train = train_df.copy()
    val = val_df.copy()
    test = test_df.copy()

    # Calculate reviewer statistics ONLY from training data
    reviewer_stats = train.groupby("reviewer_id").agg(
        reviewer_review_count=("reviewer_id", "size"),
        reviewer_avg_rating=("review_rating", "mean"),
        reviewer_rating_std=("review_rating", "std"),
        reviewer_avg_helpful=("number_of_helpful", "mean"),
    ).reset_index()

    reviewer_stats["reviewer_rating_std"] = (
        reviewer_stats["reviewer_rating_std"].fillna(0)
    )

    # Merge training-derived statistics
    train = train.merge(reviewer_stats, on="reviewer_id", how="left")
    val = val.merge(reviewer_stats, on="reviewer_id", how="left")
    test = test.merge(reviewer_stats, on="reviewer_id", how="left")

    # Reviewers unseen during training
    reviewer_feature_cols = [
        "reviewer_review_count",
        "reviewer_avg_rating",
        "reviewer_rating_std",
        "reviewer_avg_helpful"
    ]

    for col in reviewer_feature_cols:
        train[col] = train[col].fillna(0)
        val[col] = val[col].fillna(0)
        test[col] = test[col].fillna(0)

    return train, val, test


train_reviewer, val_reviewer, test_reviewer = add_reviewer_features(
    train_df,
    val_df,
    test_df
)

print("Train:", train_reviewer.shape)
print("Validation:", val_reviewer.shape)
print("Test:", test_reviewer.shape)

print("\nNew reviewer features:")
print([
    "reviewer_review_count",
    "reviewer_avg_rating",
    "reviewer_rating_std",
    "reviewer_avg_helpful"
])

Train: (23058, 28)
Validation: (4941, 28)
Test: (4941, 28)

New reviewer features:
['reviewer_review_count', 'reviewer_avg_rating', 'reviewer_rating_std', 'reviewer_avg_helpful']


In [38]:
# STEP 2: Test reviewer behavioral features

reviewer_feature_cols = [
    "reviewer_review_count",
    "reviewer_avg_rating",
    "reviewer_rating_std",
    "reviewer_avg_helpful"
]

# Start with our current best 28-feature set
base_feature_cols = basic_features + writing_style_features + sentiment_features

reviewer_features = base_feature_cols + reviewer_feature_cols

X_train_reviewer = train_reviewer[reviewer_features]
X_val_reviewer = val_reviewer[reviewer_features]

y_train = train_reviewer["target"]
y_val = val_reviewer["target"]

print("Number of features:", len(reviewer_features))
print("Training shape:", X_train_reviewer.shape)
print("Validation shape:", X_val_reviewer.shape)

reviewer_xgb = XGBClassifier(
    n_estimators=300,
    max_depth=3,
    learning_rate=0.10,
    min_child_weight=10,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

reviewer_xgb.fit(X_train_reviewer, y_train)

reviewer_val_prob = reviewer_xgb.predict_proba(X_val_reviewer)[:, 1]
reviewer_val_pred = (reviewer_val_prob >= 0.5).astype(int)

reviewer_accuracy = accuracy_score(y_val, reviewer_val_pred)
reviewer_precision = precision_score(y_val, reviewer_val_pred)
reviewer_recall = recall_score(y_val, reviewer_val_pred)
reviewer_f1 = f1_score(y_val, reviewer_val_pred)

print("\n" + "="*60)
print("XGBOOST + REVIEWER FEATURES")
print("="*60)
print(f"Accuracy : {reviewer_accuracy:.4f} ({reviewer_accuracy*100:.2f}%)")
print(f"Precision: {reviewer_precision:.4f}")
print(f"Recall   : {reviewer_recall:.4f}")
print(f"F1 Score : {reviewer_f1:.4f}")

print("\nPrevious best:")
print("Accuracy : 81.54%")
print("F1 Score : 86.88%")

NameError: name 'basic_features' is not defined

In [39]:
# Define the feature groups used in our current best model

basic_features = [
    "review_rating",
    "is_1_star",
    "is_5_star",
    "number_of_helpful",
    "has_helpful_votes",
    "photo_count",
    "has_photos",
    "review_text_length",
    "review_word_count",
    "review_title_length",
    "review_title_word_count",
    "review_year",
    "review_month",
    "review_day_of_week"
]

writing_style_features = [
    "punctuation_count",
    "exclamation_count",
    "question_count",
    "comma_count",
    "digit_count",
    "uppercase_count",
    "letter_count",
    "uppercase_ratio",
    "digit_ratio",
    "avg_word_length",
    "unique_word_ratio",
    "title_text_length_ratio"
]

sentiment_features = [
    "sentiment_polarity",
    "sentiment_subjectivity"
]

print("Basic features:", len(basic_features))
print("Writing-style features:", len(writing_style_features))
print("Sentiment features:", len(sentiment_features))

base_feature_cols = (
    basic_features
    + writing_style_features
    + sentiment_features
)

print("\nTotal base features:", len(base_feature_cols))

Basic features: 14
Writing-style features: 12
Sentiment features: 2

Total base features: 28


In [40]:
# Add reviewer behavioral features to the current best 28-feature set

reviewer_feature_cols = [
    "reviewer_review_count",
    "reviewer_avg_rating",
    "reviewer_rating_std",
    "reviewer_avg_helpful"
]

reviewer_features = base_feature_cols + reviewer_feature_cols

X_train_reviewer = train_reviewer[reviewer_features]
X_val_reviewer = val_reviewer[reviewer_features]

y_train = train_reviewer["target"]
y_val = val_reviewer["target"]

print("Total features:", len(reviewer_features))
print("Training shape:", X_train_reviewer.shape)
print("Validation shape:", X_val_reviewer.shape)

reviewer_xgb = XGBClassifier(
    n_estimators=300,
    max_depth=3,
    learning_rate=0.10,
    min_child_weight=10,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

reviewer_xgb.fit(X_train_reviewer, y_train)

reviewer_val_prob = reviewer_xgb.predict_proba(X_val_reviewer)[:, 1]
reviewer_val_pred = (reviewer_val_prob >= 0.5).astype(int)

reviewer_accuracy = accuracy_score(y_val, reviewer_val_pred)
reviewer_precision = precision_score(y_val, reviewer_val_pred)
reviewer_recall = recall_score(y_val, reviewer_val_pred)
reviewer_f1 = f1_score(y_val, reviewer_val_pred)

print("\n" + "=" * 60)
print("XGBOOST + REVIEWER FEATURES")
print("=" * 60)
print(f"Accuracy : {reviewer_accuracy:.4f} ({reviewer_accuracy*100:.2f}%)")
print(f"Precision: {reviewer_precision:.4f}")
print(f"Recall   : {reviewer_recall:.4f}")
print(f"F1 Score : {reviewer_f1:.4f}")

print("\nCurrent best:")
print("Accuracy : 81.54%")
print("F1 Score : 86.88%")

KeyError: "['is_1_star', 'is_5_star', 'has_helpful_votes', 'photo_count', 'has_photos', 'review_text_length', 'review_word_count', 'review_title_length', 'review_title_word_count', 'review_year', 'review_month', 'review_day_of_week', 'punctuation_count', 'exclamation_count', 'question_count', 'comma_count', 'digit_count', 'uppercase_count', 'letter_count', 'uppercase_ratio', 'digit_ratio', 'avg_word_length', 'unique_word_ratio', 'title_text_length_ratio', 'sentiment_polarity', 'sentiment_subjectivity'] not in index"

In [41]:
# Rebuild the 28 existing features on the reviewer DataFrames

train_reviewer = add_writing_style_features(train_reviewer)
train_reviewer = add_sentiment_features(train_reviewer)

val_reviewer = add_writing_style_features(val_reviewer)
val_reviewer = add_sentiment_features(val_reviewer)

test_reviewer = add_writing_style_features(test_reviewer)
test_reviewer = add_sentiment_features(test_reviewer)

print("Train shape:", train_reviewer.shape)
print("Validation shape:", val_reviewer.shape)
print("Test shape:", test_reviewer.shape)

missing_train = [c for c in base_feature_cols if c not in train_reviewer.columns]
missing_val = [c for c in base_feature_cols if c not in val_reviewer.columns]

print("\nMissing base features in train:", missing_train)
print("Missing base features in validation:", missing_val)

KeyError: 'review_rating'

In [ ]:
# Recreate reviewer DataFrames from the ORIGINAL splits
# This avoids modifying/corrupting the existing DataFrames.

train_reviewer, val_reviewer, test_reviewer = add_reviewer_features(
    train_df.copy(),
    val_df.copy(),
    test_df.copy()
)

print("Train columns:", len(train_reviewer.columns))
print("Validation columns:", len(val_reviewer.columns))
print("Test columns:", len(test_reviewer.columns))

print("\nreview_rating exists:")
print("Train:", "review_rating" in train_reviewer.columns)
print("Validation:", "review_rating" in val_reviewer.columns)
print("Test:", "review_rating" in test_reviewer.columns)

print("\nReviewer features:")
print([
    "reviewer_review_count",
    "reviewer_avg_rating",
    "reviewer_rating_std",
    "reviewer_avg_helpful"
])

In [43]:
# Recreate reviewer DataFrames from the ORIGINAL splits
# This avoids modifying/corrupting the existing DataFrames.

train_reviewer, val_reviewer, test_reviewer = add_reviewer_features(
    train_df.copy(),
    val_df.copy(),
    test_df.copy()
)

print("Train columns:", len(train_reviewer.columns))
print("Validation columns:", len(val_reviewer.columns))
print("Test columns:", len(test_reviewer.columns))

print("\nreview_rating exists:")
print("Train:", "review_rating" in train_reviewer.columns)
print("Validation:", "review_rating" in val_reviewer.columns)
print("Test:", "review_rating" in test_reviewer.columns)

print("\nReviewer features:")
print([
    "reviewer_review_count",
    "reviewer_avg_rating",
    "reviewer_rating_std",
    "reviewer_avg_helpful"
])

Train columns: 28
Validation columns: 28
Test columns: 28

review_rating exists:
Train: True
Validation: True
Test: True

Reviewer features:
['reviewer_review_count', 'reviewer_avg_rating', 'reviewer_rating_std', 'reviewer_avg_helpful']


In [44]:
# Build the 28 existing features + 4 reviewer features

train_reviewer_features = add_sentiment_features(train_reviewer)
val_reviewer_features = add_sentiment_features(val_reviewer)
test_reviewer_features = add_sentiment_features(test_reviewer)

reviewer_feature_cols = [
    "reviewer_review_count",
    "reviewer_avg_rating",
    "reviewer_rating_std",
    "reviewer_avg_helpful"
]

reviewer_features = base_feature_cols + reviewer_feature_cols

# Verify everything exists
missing_train = [
    c for c in reviewer_features
    if c not in train_reviewer_features.columns
]

missing_val = [
    c for c in reviewer_features
    if c not in val_reviewer_features.columns
]

print("Total features:", len(reviewer_features))
print("Train shape:", train_reviewer_features.shape)
print("Validation shape:", val_reviewer_features.shape)
print("Test shape:", test_reviewer_features.shape)

print("\nMissing train features:", missing_train)
print("Missing validation features:", missing_val)

Calculating sentiment features...
Calculating sentiment features...
Calculating sentiment features...
Total features: 32
Train shape: (23058, 28)
Validation shape: (4941, 28)
Test shape: (4941, 28)

Missing train features: ['review_rating', 'number_of_helpful', 'reviewer_review_count', 'reviewer_avg_rating', 'reviewer_rating_std', 'reviewer_avg_helpful']
Missing validation features: ['review_rating', 'number_of_helpful', 'reviewer_review_count', 'reviewer_avg_rating', 'reviewer_rating_std', 'reviewer_avg_helpful']


In [45]:
# Build the complete 32-feature set without losing existing columns

def build_reviewer_model_features(df):
    data = df.copy()

    # -------------------------------------------------
    # 1. Original behavioral features
    # -------------------------------------------------
    data = create_behavioral_features(data)

    # -------------------------------------------------
    # 2. Writing-style features
    # -------------------------------------------------
    text = data["review_text"].fillna("").astype(str)

    data["punctuation_count"] = text.str.count(r"[^\w\s]")
    data["exclamation_count"] = text.str.count("!")
    data["question_count"] = text.str.count(r"\?")
    data["comma_count"] = text.str.count(",")
    data["digit_count"] = text.str.count(r"\d")
    data["uppercase_count"] = text.str.count(r"[A-Z]")
    data["letter_count"] = text.str.count(r"[A-Za-z]")

    data["uppercase_ratio"] = (
        data["uppercase_count"] /
        data["letter_count"].replace(0, np.nan)
    ).fillna(0)

    data["digit_ratio"] = (
        data["digit_count"] /
        data["review_text_length"].replace(0, np.nan)
    ).fillna(0)

    data["avg_word_length"] = (
        data["review_text_length"] /
        data["review_word_count"].replace(0, np.nan)
    ).fillna(0)

    data["unique_word_ratio"] = text.apply(
        lambda x: (
            len(set(x.lower().split())) / len(x.lower().split())
            if len(x.lower().split()) > 0 else 0
        )
    )

    data["title_text_length_ratio"] = (
        data["review_title_length"] /
        data["review_text_length"].replace(0, np.nan)
    ).fillna(0)

    # -------------------------------------------------
    # 3. Sentiment features
    # -------------------------------------------------
    print("Calculating sentiment features...")

    sentiment_results = text.apply(
        lambda x: TextBlob(x).sentiment
    )

    data["sentiment_polarity"] = sentiment_results.apply(
        lambda x: x.polarity
    )

    data["sentiment_subjectivity"] = sentiment_results.apply(
        lambda x: x.subjectivity
    )

    return data


# Build features
train_reviewer_features = build_reviewer_model_features(train_reviewer)
val_reviewer_features = build_reviewer_model_features(val_reviewer)
test_reviewer_features = build_reviewer_model_features(test_reviewer)

# Reviewer features
reviewer_feature_cols = [
    "reviewer_review_count",
    "reviewer_avg_rating",
    "reviewer_rating_std",
    "reviewer_avg_helpful"
]

reviewer_features = (
    base_feature_cols +
    reviewer_feature_cols
)

# Check
missing_train = [
    c for c in reviewer_features
    if c not in train_reviewer_features.columns
]

missing_val = [
    c for c in reviewer_features
    if c not in val_reviewer_features.columns
]

print("\n" + "=" * 60)
print("FEATURE CHECK")
print("=" * 60)

print("Total model features:", len(reviewer_features))
print("Train shape:", train_reviewer_features.shape)
print("Validation shape:", val_reviewer_features.shape)
print("Test shape:", test_reviewer_features.shape)

print("\nMissing train features:", missing_train)
print("Missing validation features:", missing_val)

KeyError: 'review_text'

In [46]:
# Cleanly build 32 model features from the ORIGINAL dataframes

def build_32_features(df):
    data = df.copy()

    # -----------------------------
    # Basic behavioral features
    # -----------------------------
    data["review_rating"] = pd.to_numeric(
        data["review_rating"], errors="coerce"
    ).fillna(0)

    data["is_1_star"] = (data["review_rating"] == 1).astype(int)
    data["is_5_star"] = (data["review_rating"] == 5).astype(int)

    data["number_of_helpful"] = pd.to_numeric(
        data["number_of_helpful"], errors="coerce"
    ).fillna(0)

    data["has_helpful_votes"] = (
        data["number_of_helpful"] > 0
    ).astype(int)

    data["photo_count"] = data["number_of_photos"].fillna(0).apply(
        lambda x: len(x) if isinstance(x, list) else 0
    )

    data["has_photos"] = (data["photo_count"] > 0).astype(int)

    text = data["review_text"].fillna("").astype(str)
    title = data["review_title"].fillna("").astype(str)

    data["review_text_length"] = text.str.len()
    data["review_word_count"] = text.str.split().str.len()

    data["review_title_length"] = title.str.len()
    data["review_title_word_count"] = title.str.split().str.len()

    dates = pd.to_datetime(data["review_date"], errors="coerce")

    data["review_year"] = dates.dt.year.fillna(0)
    data["review_month"] = dates.dt.month.fillna(0)
    data["review_day_of_week"] = dates.dt.dayofweek.fillna(0)

    # -----------------------------
    # Writing-style features
    # -----------------------------
    data["punctuation_count"] = text.str.count(r"[^\w\s]")
    data["exclamation_count"] = text.str.count("!")
    data["question_count"] = text.str.count(r"\?")
    data["comma_count"] = text.str.count(",")
    data["digit_count"] = text.str.count(r"\d")
    data["uppercase_count"] = text.str.count(r"[A-Z]")
    data["letter_count"] = text.str.count(r"[A-Za-z]")

    data["uppercase_ratio"] = (
        data["uppercase_count"] /
        data["letter_count"].replace(0, np.nan)
    ).fillna(0)

    data["digit_ratio"] = (
        data["digit_count"] /
        data["review_text_length"].replace(0, np.nan)
    ).fillna(0)

    data["avg_word_length"] = (
        data["review_text_length"] /
        data["review_word_count"].replace(0, np.nan)
    ).fillna(0)

    data["unique_word_ratio"] = text.apply(
        lambda x: (
            len(set(x.lower().split())) / len(x.lower().split())
            if x.lower().split() else 0
        )
    )

    data["title_text_length_ratio"] = (
        data["review_title_length"] /
        data["review_text_length"].replace(0, np.nan)
    ).fillna(0)

    # -----------------------------
    # Sentiment
    # -----------------------------
    print("Calculating sentiment...")

    sentiments = text.apply(
        lambda x: TextBlob(x).sentiment
    )

    data["sentiment_polarity"] = sentiments.apply(
        lambda x: x.polarity
    )

    data["sentiment_subjectivity"] = sentiments.apply(
        lambda x: x.subjectivity
    )

    # -----------------------------
    # Return
    # -----------------------------
    return data


# Start from ORIGINAL splits
train_32 = build_32_features(train_df)
val_32 = build_32_features(val_df)
test_32 = build_32_features(test_df)

# Add reviewer statistics from training data only
reviewer_stats = train_df.groupby("reviewer_id").agg(
    reviewer_review_count=("reviewer_id", "size"),
    reviewer_avg_rating=("review_rating", "mean"),
    reviewer_rating_std=("review_rating", "std"),
    reviewer_avg_helpful=("number_of_helpful", "mean")
).reset_index()

reviewer_stats["reviewer_rating_std"] = (
    reviewer_stats["reviewer_rating_std"].fillna(0)
)

train_32 = train_32.merge(
    reviewer_stats, on="reviewer_id", how="left"
)

val_32 = val_32.merge(
    reviewer_stats, on="reviewer_id", how="left"
)

test_32 = test_32.merge(
    reviewer_stats, on="reviewer_id", how="left"
)

# Fill unseen-reviewer values
reviewer_feature_cols = [
    "reviewer_review_count",
    "reviewer_avg_rating",
    "reviewer_rating_std",
    "reviewer_avg_helpful"
]

for col in reviewer_feature_cols:
    train_32[col] = train_32[col].fillna(0)
    val_32[col] = val_32[col].fillna(0)
    test_32[col] = test_32[col].fillna(0)

reviewer_features = (
    base_feature_cols +
    reviewer_feature_cols
)

missing_train = [
    c for c in reviewer_features
    if c not in train_32.columns
]

missing_val = [
    c for c in reviewer_features
    if c not in val_32.columns
]

print("\n" + "=" * 60)
print("32-FEATURE CHECK")
print("=" * 60)

print("Total features:", len(reviewer_features))
print("Train:", train_32.shape)
print("Validation:", val_32.shape)
print("Test:", test_32.shape)

print("\nMissing train:", missing_train)
print("Missing validation:", missing_val)

Calculating sentiment...
Calculating sentiment...
Calculating sentiment...

32-FEATURE CHECK
Total features: 32
Train: (23058, 54)
Validation: (4941, 54)
Test: (4941, 54)

Missing train: []
Missing validation: []


In [47]:
# STEP 3: Test 32-feature XGBoost

X_train_32 = train_32[reviewer_features]
X_val_32 = val_32[reviewer_features]

y_train_32 = train_32["target"]
y_val_32 = val_32["target"]

print("Training:", X_train_32.shape)
print("Validation:", X_val_32.shape)

xgb_32 = XGBClassifier(
    n_estimators=300,
    max_depth=3,
    learning_rate=0.10,
    min_child_weight=10,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_32.fit(X_train_32, y_train_32)

val_32_prob = xgb_32.predict_proba(X_val_32)[:, 1]
val_32_pred = (val_32_prob >= 0.5).astype(int)

acc_32 = accuracy_score(y_val_32, val_32_pred)
prec_32 = precision_score(y_val_32, val_32_pred)
rec_32 = recall_score(y_val_32, val_32_pred)
f1_32 = f1_score(y_val_32, val_32_pred)

print("\n" + "=" * 60)
print("32-FEATURE XGBOOST")
print("=" * 60)

print(f"Accuracy : {acc_32:.4f} ({acc_32*100:.2f}%)")
print(f"Precision: {prec_32:.4f}")
print(f"Recall   : {rec_32:.4f}")
print(f"F1 Score : {f1_32:.4f}")

print("\nPrevious best:")
print("Accuracy : 81.54%")
print("F1 Score : 86.88%")

Training: (23058, 32)
Validation: (4941, 32)

32-FEATURE XGBOOST
Accuracy : 0.8407 (84.07%)
Precision: 0.9515
Recall   : 0.8040
F1 Score : 0.8716

Previous best:
Accuracy : 81.54%
F1 Score : 86.88%


In [48]:
# STEP 3: Test 32-feature XGBoost

X_train_32 = train_32[reviewer_features]
X_val_32 = val_32[reviewer_features]

y_train_32 = train_32["target"]
y_val_32 = val_32["target"]

print("Training:", X_train_32.shape)
print("Validation:", X_val_32.shape)

xgb_32 = XGBClassifier(
    n_estimators=300,
    max_depth=3,
    learning_rate=0.10,
    min_child_weight=10,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_32.fit(X_train_32, y_train_32)

val_32_prob = xgb_32.predict_proba(X_val_32)[:, 1]
val_32_pred = (val_32_prob >= 0.5).astype(int)

acc_32 = accuracy_score(y_val_32, val_32_pred)
prec_32 = precision_score(y_val_32, val_32_pred)
rec_32 = recall_score(y_val_32, val_32_pred)
f1_32 = f1_score(y_val_32, val_32_pred)

print("\n" + "=" * 60)
print("32-FEATURE XGBOOST")
print("=" * 60)

print(f"Accuracy : {acc_32:.4f} ({acc_32*100:.2f}%)")
print(f"Precision: {prec_32:.4f}")
print(f"Recall   : {rec_32:.4f}")
print(f"F1 Score : {f1_32:.4f}")

print("\nPrevious best:")
print("Accuracy : 81.54%")
print("F1 Score : 86.88%")

Training: (23058, 32)
Validation: (4941, 32)

32-FEATURE XGBOOST
Accuracy : 0.8407 (84.07%)
Precision: 0.9515
Recall   : 0.8040
F1 Score : 0.8716

Previous best:
Accuracy : 81.54%
F1 Score : 86.88%


In [49]:
# STEP 4: Feature importance of the 32-feature model

importance_df = pd.DataFrame({
    "feature": reviewer_features,
    "importance": xgb_32.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

print("=" * 60)
print("TOP 32 FEATURES")
print("=" * 60)

print(
    importance_df.to_string(index=False)
)

TOP 32 FEATURES
                feature  importance
  reviewer_review_count    0.359559
    reviewer_avg_rating    0.094748
           review_month    0.087811
           letter_count    0.084041
            review_year    0.081085
      review_word_count    0.040300
          review_rating    0.026082
    reviewer_rating_std    0.019854
         question_count    0.014781
      unique_word_ratio    0.014167
        uppercase_ratio    0.011730
        avg_word_length    0.011649
              is_5_star    0.011570
    review_title_length    0.011419
   reviewer_avg_helpful    0.010951
review_title_word_count    0.010672
            digit_ratio    0.010633
      punctuation_count    0.009692
 sentiment_subjectivity    0.009632
title_text_length_ratio    0.009233
     review_text_length    0.009014
     sentiment_polarity    0.008731
        uppercase_count    0.008572
     review_day_of_week    0.008396
            comma_count    0.008301
      number_of_helpful    0.007898
            

In [50]:
# STEP 5: Richer reviewer behavioral statistics
# All statistics are calculated from TRAINING data only.

train_source = train_df.copy()

# Make sure numeric columns are numeric
train_source["review_rating"] = pd.to_numeric(
    train_source["review_rating"], errors="coerce"
).fillna(0)

train_source["number_of_helpful"] = pd.to_numeric(
    train_source["number_of_helpful"], errors="coerce"
).fillna(0)

train_source["review_text"] = (
    train_source["review_text"].fillna("").astype(str)
)

train_source["review_length"] = train_source["review_text"].str.len()

# Temporary rating indicators
train_source["temp_5_star"] = (
    train_source["review_rating"] == 5
).astype(int)

train_source["temp_1_star"] = (
    train_source["review_rating"] == 1
).astype(int)

# Reviewer-level statistics
rich_reviewer_stats = train_source.groupby("reviewer_id").agg(
    reviewer_review_count=("reviewer_id", "size"),
    reviewer_avg_rating=("review_rating", "mean"),
    reviewer_rating_std=("review_rating", "std"),
    reviewer_avg_helpful=("number_of_helpful", "mean"),
    reviewer_avg_review_length=("review_length", "mean"),
    reviewer_5star_ratio=("temp_5_star", "mean"),
    reviewer_1star_ratio=("temp_1_star", "mean"),
    reviewer_min_rating=("review_rating", "min"),
    reviewer_max_rating=("review_rating", "max")
).reset_index()

rich_reviewer_stats["reviewer_rating_std"] = (
    rich_reviewer_stats["reviewer_rating_std"].fillna(0)
)

print("=" * 60)
print("RICH REVIEWER STATISTICS")
print("=" * 60)

print("Number of reviewer features:",
      len(rich_reviewer_stats.columns) - 1)

print("\nFeatures:")
print(list(rich_reviewer_stats.columns))

RICH REVIEWER STATISTICS
Number of reviewer features: 9

Features:
['reviewer_id', 'reviewer_review_count', 'reviewer_avg_rating', 'reviewer_rating_std', 'reviewer_avg_helpful', 'reviewer_avg_review_length', 'reviewer_5star_ratio', 'reviewer_1star_ratio', 'reviewer_min_rating', 'reviewer_max_rating']


In [51]:
# STEP 6: Merge richer reviewer statistics
# Statistics were calculated from TRAINING DATA ONLY.

rich_cols = [
    "reviewer_avg_review_length",
    "reviewer_5star_ratio",
    "reviewer_1star_ratio",
    "reviewer_min_rating",
    "reviewer_max_rating"
]

# Merge into our existing feature DataFrames
train_rich = train_32.merge(
    rich_reviewer_stats,
    on="reviewer_id",
    how="left",
    suffixes=("", "_rich")
)

val_rich = val_32.merge(
    rich_reviewer_stats,
    on="reviewer_id",
    how="left",
    suffixes=("", "_rich")
)

test_rich = test_32.merge(
    rich_reviewer_stats,
    on="reviewer_id",
    how="left",
    suffixes=("", "_rich")
)

# Fill values for reviewers not seen in training
for col in rich_cols:
    train_rich[col] = train_rich[col].fillna(0)
    val_rich[col] = val_rich[col].fillna(0)
    test_rich[col] = test_rich[col].fillna(0)

# 32 existing features + 5 genuinely new reviewer features
rich_features = reviewer_features + rich_cols

print("=" * 60)
print("RICH REVIEWER FEATURE CHECK")
print("=" * 60)

print("Total model features:", len(rich_features))

print("Train shape:", train_rich.shape)
print("Validation shape:", val_rich.shape)
print("Test shape:", test_rich.shape)

missing_train = [
    c for c in rich_features
    if c not in train_rich.columns
]

missing_val = [
    c for c in rich_features
    if c not in val_rich.columns
]

print("\nMissing train features:", missing_train)
print("Missing validation features:", missing_val)

RICH REVIEWER FEATURE CHECK
Total model features: 37
Train shape: (23058, 63)
Validation shape: (4941, 63)
Test shape: (4941, 63)

Missing train features: []
Missing validation features: []


In [52]:
# STEP 7: Test richer reviewer features

X_train_37 = train_rich[rich_features]
X_val_37 = val_rich[rich_features]

y_train_37 = train_rich["target"]
y_val_37 = val_rich["target"]

print("Training:", X_train_37.shape)
print("Validation:", X_val_37.shape)

xgb_37 = XGBClassifier(
    n_estimators=300,
    max_depth=3,
    learning_rate=0.10,
    min_child_weight=10,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_37.fit(X_train_37, y_train_37)

val_37_prob = xgb_37.predict_proba(X_val_37)[:, 1]
val_37_pred = (val_37_prob >= 0.5).astype(int)

acc_37 = accuracy_score(y_val_37, val_37_pred)
prec_37 = precision_score(y_val_37, val_37_pred)
rec_37 = recall_score(y_val_37, val_37_pred)
f1_37 = f1_score(y_val_37, val_37_pred)

print("\n" + "=" * 60)
print("37-FEATURE XGBOOST")
print("=" * 60)

print(f"Accuracy : {acc_37:.4f} ({acc_37*100:.2f}%)")
print(f"Precision: {prec_37:.4f}")
print(f"Recall   : {rec_37:.4f}")
print(f"F1 Score : {f1_37:.4f}")

print("\nPrevious:")
print("32-feature Accuracy : 84.07%")
print("32-feature F1       : 87.16%")

Training: (23058, 37)
Validation: (4941, 37)

37-FEATURE XGBOOST
Accuracy : 0.8393 (83.93%)
Precision: 0.9616
Recall   : 0.7925
F1 Score : 0.8689

Previous:
32-feature Accuracy : 84.07%
32-feature F1       : 87.16%


In [53]:
# STEP 8: Controlled XGBoost tuning on the 32-feature model

from itertools import product

param_grid = {
    "max_depth": [2, 3, 4, 5],
    "min_child_weight": [1, 5, 10, 20],
    "learning_rate": [0.03, 0.05, 0.10],
    "n_estimators": [200, 300, 500]
}

results_32 = []

X_train = train_32[reviewer_features]
X_val = val_32[reviewer_features]

y_train = train_32["target"]
y_val = val_32["target"]

total = (
    len(param_grid["max_depth"])
    * len(param_grid["min_child_weight"])
    * len(param_grid["learning_rate"])
    * len(param_grid["n_estimators"])
)

print(f"Testing {total} configurations...")

count = 0

for depth, child, lr, estimators in product(
    param_grid["max_depth"],
    param_grid["min_child_weight"],
    param_grid["learning_rate"],
    param_grid["n_estimators"]
):
    
    model = XGBClassifier(
        n_estimators=estimators,
        max_depth=depth,
        learning_rate=lr,
        min_child_weight=child,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    pred = model.predict(X_val)

    accuracy = accuracy_score(y_val, pred)
    precision = precision_score(y_val, pred)
    recall = recall_score(y_val, pred)
    f1 = f1_score(y_val, pred)

    results_32.append({
        "max_depth": depth,
        "min_child_weight": child,
        "learning_rate": lr,
        "n_estimators": estimators,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

    count += 1

    if count % 10 == 0:
        print(f"Completed {count}/{total}")

results_32_df = pd.DataFrame(results_32)

results_32_df = results_32_df.sort_values(
    "accuracy",
    ascending=False
).reset_index(drop=True)

print("\n" + "=" * 70)
print("TOP 15 CONFIGURATIONS")
print("=" * 70)

print(
    results_32_df.head(15).to_string(index=False)
)

Testing 144 configurations...
Completed 10/144
Completed 20/144
Completed 30/144
Completed 40/144
Completed 50/144
Completed 60/144
Completed 70/144
Completed 80/144
Completed 90/144
Completed 100/144
Completed 110/144
Completed 120/144
Completed 130/144
Completed 140/144

TOP 15 CONFIGURATIONS
 max_depth  min_child_weight  learning_rate  n_estimators  accuracy  precision   recall       f1
         3                 1           0.05           500  0.842947   0.954302 0.804878 0.873244
         4                10           0.10           200  0.842744   0.952026 0.806685 0.873350
         3                10           0.10           200  0.842137   0.952296 0.805480 0.872757
         4                20           0.10           200  0.841935   0.952279 0.805179 0.872573
         5                 5           0.05           500  0.841732   0.952262 0.804878 0.872389
         4                 1           0.03           500  0.841732   0.952262 0.804878 0.872389
         4               

In [54]:
# STEP 9: Threshold tuning for the best 32-feature XGBoost

best_xgb_32 = XGBClassifier(
    n_estimators=500,
    max_depth=3,
    learning_rate=0.05,
    min_child_weight=1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

best_xgb_32.fit(
    train_32[reviewer_features],
    train_32["target"]
)

best_val_prob = best_xgb_32.predict_proba(
    val_32[reviewer_features]
)[:, 1]

threshold_results = []

for threshold in np.arange(0.30, 0.71, 0.01):

    pred = (best_val_prob >= threshold).astype(int)

    threshold_results.append({
        "threshold": threshold,
        "accuracy": accuracy_score(y_val, pred),
        "precision": precision_score(y_val, pred),
        "recall": recall_score(y_val, pred),
        "f1": f1_score(y_val, pred)
    })

threshold_df = pd.DataFrame(threshold_results)

threshold_df = threshold_df.sort_values(
    "accuracy",
    ascending=False
).reset_index(drop=True)

print("=" * 70)
print("TOP THRESHOLDS BY VALIDATION ACCURACY")
print("=" * 70)

print(
    threshold_df.head(15).to_string(index=False)
)

TOP THRESHOLDS BY VALIDATION ACCURACY
 threshold  accuracy  precision   recall       f1
      0.30  0.870876   0.919875 0.884974 0.902087
      0.31  0.869257   0.921260 0.880759 0.900554
      0.32  0.866829   0.923370 0.874435 0.898237
      0.33  0.864805   0.925297 0.869015 0.896273
      0.34  0.863186   0.926751 0.864800 0.894704
      0.35  0.862376   0.930551 0.859380 0.893550
      0.36  0.859947   0.931408 0.854562 0.891332
      0.37  0.859138   0.933598 0.850949 0.890359
      0.38  0.857923   0.935774 0.846733 0.889029
      0.39  0.857519   0.938065 0.843722 0.888396
      0.40  0.856709   0.941534 0.838904 0.887261
      0.41  0.855495   0.942314 0.836194 0.886088
      0.42  0.854685   0.943744 0.833484 0.885193
      0.43  0.852864   0.944483 0.829871 0.883475
      0.44  0.850030   0.945135 0.824752 0.880849


In [55]:
# STEP 10: Fine-grained threshold search around the promising region

fine_thresholds = np.arange(0.20, 0.351, 0.005)

fine_results = []

for threshold in fine_thresholds:

    pred = (best_val_prob >= threshold).astype(int)

    fine_results.append({
        "threshold": threshold,
        "accuracy": accuracy_score(y_val, pred),
        "precision": precision_score(y_val, pred),
        "recall": recall_score(y_val, pred),
        "f1": f1_score(y_val, pred)
    })

fine_threshold_df = pd.DataFrame(fine_results)

fine_threshold_df = fine_threshold_df.sort_values(
    "accuracy",
    ascending=False
).reset_index(drop=True)

print("=" * 70)
print("FINE-GRAINED THRESHOLD SEARCH")
print("=" * 70)

print(
    fine_threshold_df.head(15).to_string(index=False)
)

FINE-GRAINED THRESHOLD SEARCH
 threshold  accuracy  precision   recall       f1
     0.210  0.874924   0.891399 0.926829 0.908769
     0.220  0.874722   0.894568 0.922313 0.908228
     0.215  0.874722   0.892733 0.924721 0.908445
     0.255  0.874722   0.907419 0.906052 0.906735
     0.230  0.874317   0.897995 0.917194 0.907493
     0.265  0.874317   0.910085 0.902138 0.906094
     0.205  0.874115   0.889017 0.928636 0.908395
     0.240  0.874115   0.901040 0.912978 0.906970
     0.225  0.874115   0.896795 0.918398 0.907468
     0.260  0.873912   0.908293 0.903643 0.905962
     0.235  0.873912   0.899585 0.914484 0.906973
     0.250  0.873710   0.904590 0.907859 0.906222
     0.200  0.873507   0.886468 0.931045 0.908210
     0.245  0.873305   0.902359 0.909967 0.906147
     0.270  0.873103   0.910921 0.899127 0.904986


In [56]:
# STEP 11: Test class weighting

class_weights = [0.7, 0.85, 1.0, 1.15, 1.3]

class_weight_results = []

X_train = train_32[reviewer_features]
X_val = val_32[reviewer_features]

y_train = train_32["target"]
y_val = val_32["target"]

for weight in class_weights:

    model = XGBClassifier(
        n_estimators=500,
        max_depth=3,
        learning_rate=0.05,
        min_child_weight=1,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=weight,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    prob = model.predict_proba(X_val)[:, 1]

    # Use the currently promising threshold
    pred = (prob >= 0.21).astype(int)

    class_weight_results.append({
        "scale_pos_weight": weight,
        "accuracy": accuracy_score(y_val, pred),
        "precision": precision_score(y_val, pred),
        "recall": recall_score(y_val, pred),
        "f1": f1_score(y_val, pred)
    })

class_weight_df = pd.DataFrame(class_weight_results)

print("=" * 70)
print("CLASS WEIGHT EXPERIMENT")
print("=" * 70)

print(class_weight_df.to_string(index=False))

CLASS WEIGHT EXPERIMENT
 scale_pos_weight  accuracy  precision   recall       f1
             0.70  0.873103   0.910671 0.899428 0.905014
             0.85  0.875531   0.902200 0.913881 0.908003
             1.00  0.874924   0.891399 0.926829 0.908769
             1.15  0.870269   0.878317 0.936766 0.906601
             1.30  0.868650   0.871938 0.943089 0.906119
